## 1. Importar

In [1]:
import time
from pathlib import Path

import pandas as pd
import requests

PASTA_RESULTADOS = Path.cwd() / "Resultados"
PASTA_RESULTADOS.mkdir(exist_ok=True)

print("vou gravar em:", PASTA_RESULTADOS)

vou gravar em: c:\Users\loude\Downloads\Projeto_TCC\Projeto_TCC\ColetarDados\Resultados


In [2]:
# dois servidores com os mesmos dados do OpenStreetMap: se um estiver lotado, tenta o outro
SERVIDORES = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]

# identifica quem está consultando — é a regra de uso do servidor público
CABECALHOS = {
    'User-Agent': 'TCC-UFFS-PALOP/1.0 (pesquisa academica; recomendacao de exportacao)',
    'Content-Type': 'text/plain; charset=utf-8',
}

# respostas que querem dizer "servidor ocupado ou com falha passageira": vale tentar de novo
OCUPADO = (429, 500, 502, 503, 504)
TENTATIVAS = 5


def consultar_overpass(consulta):
    """Manda uma consulta Overpass QL. Devolve (resposta, aviso_de_erro).

    Se o servidor estiver ocupado, espera e tenta de novo, alternando entre os servidores.
    """
    aviso = ""
    for tentativa in range(TENTATIVAS):
        if tentativa > 0:
            espera = 15 * 2 ** (tentativa - 1)          # 15, 30, 60, 120 segundos
            print("   servidor ocupado; tento de novo em", espera, "s")
            time.sleep(espera)

        endereco = SERVIDORES[tentativa % len(SERVIDORES)]
        try:
            resposta = requests.post(endereco, data=consulta.encode('utf-8'),
                                     headers=CABECALHOS, timeout=180)
        except Exception as erro:
            aviso = "não foi possível falar com o servidor: " + str(erro)
            continue

        if resposta.status_code in OCUPADO:
            aviso = "servidor ocupado (" + str(resposta.status_code) + ") em todas as tentativas"
            continue
        if resposta.status_code != 200:
            return {}, ("o servidor respondeu " + str(resposta.status_code) + ": "
                        + resposta.text[:200])

        try:
            return resposta.json(), ""
        except Exception:
            aviso = "a resposta não veio em JSON: " + resposta.text[:200]

    return {}, aviso


print("função pronta")

função pronta


In [3]:
PAISES_ISO = {
    'Angola':              'AO',
    'Cabo Verde':          'CV',
    'Guiné-Bissau':        'GW',
    'Moçambique':          'MZ',
    'São Tomé e Príncipe': 'ST',
}

# palavras que indicam alimento no nome — em português e inglês (Moçambique usa muito o inglês)
PALAVRAS_ALIMENTO = ['aliment', 'food', 'mercearia', 'viveres', 'víveres', 'cash and carry',
                     'cash & carry', 'cash&carry', 'grocer', 'supermerc', 'hipermerc']

# sub-etiquetas de atacado que são do setor alimentício
ATACADO_ALIMENTO = ['food', 'beverages', 'grocery']


def montar_consulta(iso):
    return (
        '[out:json][timeout:120];\n'
        'area["ISO3166-1"="' + iso + '"]->.p;\n'
        '(\n'
        '  nwr["shop"="wholesale"]["wholesale"~"' + '|'.join(ATACADO_ALIMENTO) + '",i](area.p);\n'
        '  nwr["shop"~"^(supermarket|food)$"](area.p);\n'
        '  nwr["name"~"' + '|'.join(PALAVRAS_ALIMENTO) + '",i](area.p);\n'
        ');\n'
        'out tags center 1000;'
    )


print(montar_consulta('AO'))

[out:json][timeout:120];
area["ISO3166-1"="AO"]->.p;
(
  nwr["shop"="wholesale"]["wholesale"~"food|beverages|grocery",i](area.p);
  nwr["shop"~"^(supermarket|food)$"](area.p);
  nwr["name"~"aliment|food|mercearia|viveres|víveres|cash and carry|cash & carry|cash&carry|grocer|supermerc|hipermerc",i](area.p);
);
out tags center 1000;


In [4]:
encontrados = []
avisos = []

for pais in PAISES_ISO:
    iso = PAISES_ISO[pais]
    resposta, aviso = consultar_overpass(montar_consulta(iso))

    if aviso != "":
        avisos.append(pais + ": " + aviso)
        print("x", pais, "->", aviso)
        continue

    elementos = resposta.get('elements', [])
    for elemento in elementos:
        etiquetas = elemento.get('tags', {})
        nome = etiquetas.get('name', '')
        if nome == "":
            continue                      # sem nome não serve para nada

        encontrados.append({
            'osm_id':   str(elemento.get('type', '')) + '/' + str(elemento.get('id', '')),
            'nome':     nome,
            'pais':     pais,
            'shop':     etiquetas.get('shop', ''),
            'office':   etiquetas.get('office', ''),
            'wholesale': etiquetas.get('wholesale', ''),
            'amenity':  etiquetas.get('amenity', ''),
            'site':     etiquetas.get('website', '') or etiquetas.get('contact:website', ''),
            'telefone': etiquetas.get('phone', '') or etiquetas.get('contact:phone', ''),
            'rua':      etiquetas.get('addr:street', ''),
            'cidade':   etiquetas.get('addr:city', ''),
        })

    print("ok", pais.ljust(22), "->", len(elementos), "objetos")
    time.sleep(5)

print()
print("total bruto:", len(encontrados), "empresas com nome")

   servidor ocupado; tento de novo em 15 s
   servidor ocupado; tento de novo em 30 s
ok Angola                 -> 290 objetos
ok Cabo Verde             -> 187 objetos
ok Guiné-Bissau           -> 25 objetos
   servidor ocupado; tento de novo em 15 s
   servidor ocupado; tento de novo em 30 s
ok Moçambique             -> 296 objetos
   servidor ocupado; tento de novo em 15 s
   servidor ocupado; tento de novo em 30 s
ok São Tomé e Príncipe    -> 10 objetos

total bruto: 739 empresas com nome


In [5]:
def e_atividade_alimentar(empresa):
    if empresa['shop'] in ('supermarket', 'food'):
        return True
    if empresa['shop'] == 'wholesale':
        for valor in ATACADO_ALIMENTO:
            if valor in empresa['wholesale'].lower():
                return True
    return False


def nome_de_alimento(empresa):
    nome_minusculo = empresa['nome'].lower()
    for palavra in PALAVRAS_ALIMENTO:
        if palavra in nome_minusculo:
            return True
    return False


por_id = {}
ordem = []

for empresa in encontrados:
    chave = empresa['osm_id']
    if chave in por_id:
        continue
    por_id[chave] = empresa
    ordem.append(chave)

unicas = []
for chave in ordem:
    empresa = por_id[chave]
    if empresa['amenity'] != '':
        continue                          # restaurante, café, bar: não é comprador de importação
    if e_atividade_alimentar(empresa) or nome_de_alimento(empresa):
        unicas.append(empresa)

print("de", len(encontrados), "brutos ->", len(por_id), "sem repetição ->",
      len(unicas), "do setor alimentício")

de 739 brutos -> 739 sem repetição -> 693 do setor alimentício


In [6]:
for empresa in unicas:
    empresa['fonte'] = 'OpenStreetMap — Overpass API'

todas = unicas
print(len(todas), "empresas do OpenStreetMap")

693 empresas do OpenStreetMap


In [7]:
PALAVRAS_ATACADO = ['atacad', 'distribu', 'import', 'grossist', 'cash', 'armaz']


def pontuar(empresa):
    pontos = 0
    motivos = []

    if empresa['shop'] in ('wholesale', 'trade') or empresa['office'] == 'wholesale':
        pontos = pontos + 3
        motivos.append('registada como grossista no OpenStreetMap')
    elif empresa['shop'] == 'supermarket':
        pontos = pontos + 1
        motivos.append('registada como supermercado')

    nome_minusculo = empresa['nome'].lower()
    for palavra in PALAVRAS_ATACADO:
        if palavra in nome_minusculo:
            pontos = pontos + 2
            motivos.append('nome indica atacado ou importação')
            break

    if empresa['site'] != '':
        pontos = pontos + 2
        motivos.append('tem site')

    if empresa['telefone'] != '' or empresa['rua'] != '':
        pontos = pontos + 1
        motivos.append('cadastro com contato ou endereço')

    return pontos, '; '.join(motivos)


for empresa in todas:
    empresa['pontos'], empresa['motivos'] = pontuar(empresa)

tabela = pd.DataFrame(todas)
print(len(tabela), "empresas pontuadas")

693 empresas pontuadas


In [8]:
import re
import unicodedata


def nome_padrao(nome):
    # minúsculas, sem acento, & vira e, só letras e números
    sem_acento = unicodedata.normalize('NFKD', nome).encode('ascii', 'ignore').decode()
    texto = sem_acento.lower().replace('&', 'e')
    return re.sub(r'[^a-z0-9]', '', texto)


# nomes que são só o tipo de loja, já escritos no padrão acima
NOMES_GENERICOS = ['mercearia', 'minimercearia', 'minemercearia', 'supermercado', 'minimercado',
                   'hipermercado', 'mercado', 'mercadocentral', 'mercadomunicipal', 'loja',
                   'lojaalimentar', 'armazem', 'cashecarry', 'cashandcarry']


def completude(empresa):
    # desempate entre lojas da mesma rede: mais pontos, depois cadastro mais completo
    return (empresa['pontos'], empresa['site'] != '', empresa['telefone'] != '', empresa['rua'] != '')


# 1) separa os genéricos e agrupa o resto por (nome padrão, país)
genericos = []
grupos = {}
ordem_grupos = []

for empresa in todas:
    chave_nome = nome_padrao(empresa['nome'])
    if chave_nome in NOMES_GENERICOS:
        genericos.append(empresa)
        continue

    chave = (chave_nome, empresa['pais'])
    if chave not in grupos:
        grupos[chave] = []
        ordem_grupos.append(chave)
    grupos[chave].append(empresa)


# 2) de cada grupo fica uma empresa, a mais completa, com o número de unidades e as cidades
juntadas = []
for chave in ordem_grupos:
    lojas = grupos[chave]
    melhor = max(lojas, key=completude)

    cidades = []
    for loja in lojas:
        if loja['cidade'] != '' and loja['cidade'] not in cidades:
            cidades.append(loja['cidade'])

    melhor['unidades'] = len(lojas)
    melhor['cidades'] = ', '.join(sorted(cidades))
    if len(lojas) > 1:
        melhor['motivos'] = (melhor['motivos'] + '; aparece ' + str(len(lojas))
                             + ' vezes no mapa (rede de lojas)').strip('; ')
    juntadas.append(melhor)

redes = [e for e in juntadas if e['unidades'] > 1]

print(len(todas), "registos")
print("  -", len(genericos), "com nome genérico (saem)")
print("  -", len(todas) - len(genericos) - len(juntadas), "lojas da mesma rede (viram uma linha só)")
print("  =", len(juntadas), "empresas")
print()
print("redes juntadas:", len(redes))
for empresa in sorted(redes, key=lambda e: -e['unidades'])[:15]:
    print("  ", empresa['nome'].ljust(28), empresa['pais'].ljust(22), empresa['unidades'], "registos")

todas = juntadas
tabela = pd.DataFrame(todas)

693 registos
  - 87 com nome genérico (saem)
  - 124 lojas da mesma rede (viram uma linha só)
  = 482 empresas

redes juntadas: 34
   NossoSuper                   Angola                 15 registos
   Angomart                     Angola                 14 registos
   Shoprite                     Moçambique             14 registos
   Shoprite                     Angola                 13 registos
   Maxi                         Angola                 9 registos
   Kibabo                       Angola                 7 registos
   Calú & Ângela                Cabo Verde             7 registos
   Spar                         Moçambique             6 registos
   Casa dos Frescos             Angola                 5 registos
   Mercearia Dona Katia         Moçambique             5 registos
   Pomobel                      Angola                 4 registos
   Alimenta Angola              Angola                 4 registos
   Recheio                      Moçambique             4 registos
   Kero

In [9]:
NAO_ALIMENTO = ['sanal piscinas', 'arco-íris cosméticos', 'tabacaria matondo',
                'centro de saúde materno infantil do benfica',
                'administração municipal do kilamba kiaxi', 'world food program',
                'angoship', 'sinase', 'angola códigos de barra',
                'unavi prestação de serviços', 'ta quang binh - prestação de serviço',
                'loja de materias de construção - tchioco',
                'supermercado materiais de construção piva ferro fábrica',
                # os mesmos nomes, como estão escritos no OpenStreetMap
                'administração municípal do kilamba kiaxi,\ndirecção municipal de fiscalização e '
                'inspeção das actividades económicas e segurança alimentar',
                'sinase - consultoria, auditoria e formação, qualidade, ambiente, segurança '
                '(iso 9001;14001;ohas 18001), haccp e segurança alimentar (iso 22000)',
                'ta quang binh - comercio, prestação de serviço']

antes = len(todas)
todas = [e for e in todas if e['nome'].strip().lower() not in NAO_ALIMENTO]
print("retiradas por não serem de alimentos:", antes - len(todas))
tabela = pd.DataFrame(todas)

retiradas por não serem de alimentos: 13


In [ ]:
PALAVRAS_DISTRIBUIDOR = ['atacad', 'distribu', 'grossist', 'grosso', 'cash', 'armaz', 'import']


def descobrir_canal(empresa):
    # distribuidor: marcado como atacado no OpenStreetMap
    if empresa['shop'] == 'wholesale':
        return 'distribuidor'
    # ambos: nome de atacado (cash & carry, armazém, venda a grosso...),
    # mas marcado como loja no mapa -> vende para revenda e para o consumidor
    nome = empresa['nome'].lower()
    for palavra in PALAVRAS_DISTRIBUIDOR:
        if palavra in nome:
            return 'ambos'
    # o resto (supermercado, mercearia, loja) vende ao consumidor
    return 'varejo'


def descrever(empresa):
    partes = []
    if empresa['shop'] != '':
        partes.append('shop=' + empresa['shop'])
    if empresa['office'] != '':
        partes.append('office=' + empresa['office'])
    if len(partes) == 0:
        return 'Identificada pelo nome no OpenStreetMap'
    return 'OpenStreetMap: ' + ', '.join(partes)


def link_do_mapa(nome, pais):
    busca = (str(nome) + ' ' + str(pais)).replace(' ', '+').replace('&', 'e')
    return 'https://www.google.com/maps/search/?api=1&query=' + busca


tabela['canal'] = [descobrir_canal(e) for e in todas]
tabela['tipo'] = [descrever(e) for e in todas]
tabela['link_mapa'] = [link_do_mapa(e['nome'], e['pais']) for e in todas]
tabela['endereco'] = [(e['rua'] + ', ' + e['cidade']).strip(', ') for e in todas]

print(tabela[['nome', 'pais', 'canal', 'pontos']].head(10).to_string(index=False))

In [11]:
ARQ_RANKING = PASTA_RESULTADOS / "ranking_final.csv"
forca_do_pais = {}

if ARQ_RANKING.exists():
    ranking = pd.read_csv(ARQ_RANKING, dtype={'Código NCM': str}, encoding='utf-8-sig')
    em_destaque = ranking[ranking['posicao_final'] <= 2]
    contagem = em_destaque['Países'].value_counts()
    for pais in PAISES_ISO:
        forca_do_pais[pais] = int(contagem.get(pais, 0))
    for pais in sorted(forca_do_pais, key=lambda p: -forca_do_pais[p]):
        print(pais.ljust(22), '->', forca_do_pais[pais], 'vezes em 1º ou 2º')
else:
    for pais in PAISES_ISO:
        forca_do_pais[pais] = 0
    print('Não achei ranking_final.csv. Rode Dados_Comex.ipynb e Recomendar.ipynb antes.')

Angola                 -> 39 vezes em 1º ou 2º
Cabo Verde             -> 29 vezes em 1º ou 2º
Moçambique             -> 15 vezes em 1º ou 2º
São Tomé e Príncipe    -> 12 vezes em 1º ou 2º
Guiné-Bissau           -> 5 vezes em 1º ou 2º


In [12]:
if len(tabela) > 0:
    tabela['forca_do_pais'] = [forca_do_pais.get(p, 0) for p in tabela['pais']]
    tabela = tabela.sort_values(['forca_do_pais', 'pontos'],
                                ascending=[False, False]).reset_index(drop=True)

    tabela['place_id'] = tabela['osm_id']
    tabela['nota'] = ''
    tabela['avaliacoes'] = ''

    COLUNAS = ['place_id', 'nome', 'pais', 'canal', 'tipo', 'endereco', 'telefone', 'site',
               'nota', 'avaliacoes', 'link_mapa', 'pontos', 'motivos', 'unidades', 'cidades',
               'forca_do_pais', 'fonte']
    tabela = tabela[COLUNAS]

    print(tabela[['nome', 'pais', 'pontos', 'unidades', 'forca_do_pais']].head(20).to_string(index=False))
else:
    print('tabela vazia')

                                                                                         nome   pais  pontos  unidades  forca_do_pais
                                                                  Armazém de Bens Alimentares Angola       6         2             39
                                                         MAFCOM (Angola) Comércio Geral, Lda. Angola       6         1             39
                                                                        Mercadão Cash & Carry Angola       6         1             39
                                                                  Nossa Casa - Cash and Carry Angola       6         1             39
                                                                               Maxi - Maianga Angola       4         1             39
                                                                                         Kero Angola       4         3             39
                                                              

## 9. Gravar e conferir

In [13]:
if len(tabela) > 0:
    destino = PASTA_RESULTADOS / "distribuidores.csv"
    tabela.to_csv(destino, index=False, encoding='utf-8-sig')
    print("gravei", len(tabela), "candidatos em:", destino)
else:
    print("Não gravei nada: a tabela está vazia. Confira os avisos da parte 4.")

if len(avisos) > 0:
    print()
    print("consultas que falharam:")
    for aviso in avisos:
        print(" -", aviso)

gravei 469 candidatos em: c:\Users\loude\Downloads\Projeto_TCC\Projeto_TCC\ColetarDados\Resultados\distribuidores.csv


In [14]:
if len(tabela) > 0:
    print("POR PAÍS (na ordem do ranking)")
    for pais in sorted(forca_do_pais, key=lambda p: -forca_do_pais[p]):
        bloco = tabela[tabela['pais'] == pais]
        print("  ", pais.ljust(22), "força", str(forca_do_pais[pais]).rjust(3),
              "->", len(bloco), "candidatos")

    print()
    print("POR FONTE")
    print(tabela['fonte'].value_counts().to_string())

    print()
    print("OS DEZ PRIMEIROS")
    for i in range(min(10, len(tabela))):
        linha = tabela.iloc[i]
        print(" ", str(i + 1).rjust(2) + ".", linha['nome'], "|", linha['pais'],
              "|", linha['pontos'], "pontos")
        print("      ", linha['motivos'])

POR PAÍS (na ordem do ranking)
   Angola                 força  39 -> 160 candidatos
   Cabo Verde             força  29 -> 125 candidatos
   Moçambique             força  15 -> 155 candidatos
   São Tomé e Príncipe    força  12 -> 9 candidatos
   Guiné-Bissau           força   5 -> 20 candidatos

POR FONTE
fonte
OpenStreetMap — Overpass API    469

OS DEZ PRIMEIROS
   1. Armazém de Bens Alimentares | Angola | 6 pontos
       registada como grossista no OpenStreetMap; nome indica atacado ou importação; cadastro com contato ou endereço; aparece 2 vezes no mapa (rede de lojas)
   2. MAFCOM (Angola) Comércio Geral, Lda. | Angola | 6 pontos
       registada como grossista no OpenStreetMap; tem site; cadastro com contato ou endereço
   3. Mercadão Cash & Carry | Angola | 6 pontos
       registada como supermercado; nome indica atacado ou importação; tem site; cadastro com contato ou endereço
   4. Nossa Casa - Cash and Carry | Angola | 6 pontos
       registada como grossista no OpenStreetM